In [8]:
import cv2
import numpy as np
import glob
import os

CARD_WIDTH_MM = 85.60
CARD_HEIGHT_MM = 53.98

def get_camera_intrinsics(checkerboard_dir, grid_size, square_size_mm):
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
    objp = np.zeros((grid_size[0] * grid_size[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:grid_size[0], 0:grid_size[1]].T.reshape(-1, 2) * square_size_mm
    
    objpoints = []
    imgpoints = []
    
    images = glob.glob(f"{checkerboard_dir}/*.jpg")
    total_images = len(images)
    print(f"[STEP 1] Found {total_images} checkerboard images in target directory.")
    
    success_count = 0
    for idx, fname in enumerate(images, 1):
        img = cv2.imread(fname)
        if img is None:
            continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        ret, corners = cv2.findChessboardCorners(gray, grid_size, None)
        
        if ret:
            success_count += 1
            objpoints.append(objp)
            corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
            imgpoints.append(corners2)
            print(f" -> Grid detected successfully in checkerboard image {idx}/{total_images}")
        else:
            print(f" -> WARNING: Could not find grid pattern in checkerboard image {idx}/{total_images}")
            
    if len(objpoints) == 0:
        raise ValueError(f"CRITICAL: OpenCV could not detect the {grid_size} grid vertices in any checkerboard images.")
        
    print(f"[STEP 2] Computing intrinsic matrix using {success_count} successful calibration frames...")
    _, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)
    
    total_error = 0
    total_points = 0
    for i in range(len(objpoints)):
        imgpoints2, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], mtx, dist)
        error = cv2.norm(imgpoints[i], imgpoints2, cv2.NORM_L2)
        total_error += (error ** 2)
        total_points += len(imgpoints[i])
        
    reprojection_error = np.sqrt(total_error / total_points)
    return mtx, dist, reprojection_error

def undistort_image(img, mtx, dist):
    h, w = img.shape[:2]
    newcameramtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w, h), 1, (w, h))
    dst = cv2.undistort(img, mtx, dist, None, newcameramtx)
    x, y, roi_w, roi_h = roi
    return dst[y:y+roi_h, x:x+roi_w]

def get_sadapay_hsv_bounds():
    return [80, 70, 50], [105, 255, 255]

def detect_sadapay_card(img, lower_hsv, upper_hsv):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array(lower_hsv), np.array(upper_hsv))
    
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    valid_contours = [c for c in contours if cv2.contourArea(c) > 500]
    if not valid_contours:
        return None
        
    largest_contour = max(valid_contours, key=cv2.contourArea)
    _, _, w, h = cv2.boundingRect(largest_contour)
    
    return w, h

def calculate_dynamic_ppm(img, use_width=True):
    lower_hsv, upper_hsv = get_sadapay_hsv_bounds()
    pixel_dims = detect_sadapay_card(img, lower_hsv, upper_hsv)
    
    if pixel_dims:
        pixel_w, pixel_h = pixel_dims
        if use_width:
            return pixel_w / CARD_WIDTH_MM
        return pixel_h / CARD_HEIGHT_MM
        
    return None

def process_calibration_pipeline(checkerboard_dir, target_images_dir, grid_size, square_size_mm):
    print("================================================================================")
    print("LAUNCHING AUTOMATED CAMERA CALIBRATION AND SCALE EXTRACTION PIPELINE")
    print("================================================================================")
    
    mtx, dist, reproj_err = get_camera_intrinsics(checkerboard_dir, grid_size, square_size_mm)
    print(f"[STEP 3] Camera distortion parameters successfully extracted. RMS Error: {reproj_err:.4f} pixels")
    
    target_images = glob.glob(f"{target_images_dir}/*.jpg")
    total_targets = len(target_images)
    print(f"[STEP 4] Scanning {total_targets} target training images for scale calculation...")
    
    ratios = []
    card_detected_count = 0
    
    for idx, path in enumerate(target_images, 1):
        img = cv2.imread(path)
        if img is None:
            continue
            
        undistorted_img = undistort_image(img, mtx, dist)
        ppm = calculate_dynamic_ppm(undistorted_img, use_width=True)
        
        filename = os.path.basename(path)
        if ppm is not None:
            ratios.append(ppm)
            card_detected_count += 1
            print(f" -> Image {idx}/{total_targets} ({filename}): SadaPay Card detected. Current PPM: {ppm:.4f}")
        else:
            print(f" -> Image {idx}/{total_targets} ({filename}): WARNING - Turquoise card not found. Skipping ratio logging.")
            
    print("================================================================================")
    print("PIPELINE PROCESSING COMPLETE")
    print("================================================================================")
    print(f"Total target images scanned: {total_targets}")
    print(f"Successful card detections:  {card_detected_count}/{total_targets}")
    
    if ratios:
        final_ppm = np.mean(ratios)
        print(f"Calculated Global Average Scale Factor.")
        return mtx, dist, final_ppm, reproj_err
    return mtx, dist, None, reproj_err

CHECKERBOARD_DIR = "/kaggle/input/datasets/muhammadunssrahim/checkboard-updated/CheckboardUpdated"
TARGET_IMAGES_DIR = "/kaggle/input/datasets/muhammadunssrahim/teabox/train"
GRID_SIZE = (13, 9)
SQUARE_SIZE_MM = 20.0

mtx, dist, final_ppm, reprojection_error = process_calibration_pipeline(CHECKERBOARD_DIR, TARGET_IMAGES_DIR, GRID_SIZE, SQUARE_SIZE_MM)

print("\n[FINAL OUTPUT VALS FOR INFERENCE ENGINE]")
print(f"REPROJECTION_ERROR (RMS): {reprojection_error:.6f} pixels")
print(f"PIXELS_PER_MM: {final_ppm}")
print(f"CAMERA_MATRIX:\n{mtx}")
print(f"DISTORTION_COEFFICIENTS:\n{dist}")

LAUNCHING AUTOMATED CAMERA CALIBRATION AND SCALE EXTRACTION PIPELINE
[STEP 1] Found 25 checkerboard images in target directory.
 -> Grid detected successfully in checkerboard image 1/25
 -> WARNING: Could not find grid pattern in checkerboard image 2/25
 -> WARNING: Could not find grid pattern in checkerboard image 3/25
 -> WARNING: Could not find grid pattern in checkerboard image 4/25
 -> Grid detected successfully in checkerboard image 5/25
 -> WARNING: Could not find grid pattern in checkerboard image 6/25
 -> Grid detected successfully in checkerboard image 7/25
 -> WARNING: Could not find grid pattern in checkerboard image 8/25
 -> WARNING: Could not find grid pattern in checkerboard image 9/25
 -> Grid detected successfully in checkerboard image 10/25
 -> WARNING: Could not find grid pattern in checkerboard image 11/25
 -> Grid detected successfully in checkerboard image 12/25
 -> WARNING: Could not find grid pattern in checkerboard image 13/25
 -> WARNING: Could not find grid p